# Mini Project 1 — Analysis Notebook

**Your name:** Aleigha Mattison 
**Dataset:**  Zillow Housing Data
**Date:**  5.20.26

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [19]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px


from pathlib import Path
import os

def _find_data_dir():
    """Use the folder that contains this notebook's CSV files."""
    cwd = Path.cwd()
    for candidate in (cwd, cwd / 'Mini project 1'):
        if (candidate / 'listings.csv').exists():
            return candidate
    return cwd

os.chdir(_find_data_dir())
print(f'Working directory: {Path.cwd().resolve()}')

def data_quality_check(csv_path=None):
    """Print shape, missing values, dtypes, outliers, and string issues for df."""
    global df
    if csv_path is not None:
        df = pd.read_csv(csv_path)
        print(f'Loaded: {csv_path}  shape: {df.shape}')
    elif 'df' not in globals():
        raise RuntimeError(
            'df is not defined. Run the Setup cell first, then either the '
            'pd.read_csv(...) cell above or call data_quality_check(csv_path=...).'
        )

    print('Data quality check')
    print('shape:', df.shape)
    print('duplicate cols:', any(df.columns.duplicated()))
    missing = df.isna().sum()
    print('total missing values:', missing.sum())
    print('columns with missing values:', (missing > 0).sum())
    print('dtypes:')
    print(df.dtypes.value_counts().to_dict())
    nums = df.select_dtypes(include=['number'])
    if not nums.empty:
        z = (nums - nums.mean()) / nums.std(ddof=0)
        print('numeric outliers >3σ:', int((z.abs() > 3).sum().sum()))
    else:
        print('numeric outliers >3σ: none')
    strings = df.select_dtypes(include=['object', 'string'])
    if not strings.empty:
        bad = []
        for col in strings.columns:
            if strings[col].dropna().astype(str).str.contains(r'^\s+|\s+$|\s{2,}|\r|\n', na=False).any():
                bad.append(col)
        print('string formatting issues:', bad if bad else 'none')
    else:
        print('string formatting issues: none')

print("Setup complete.")

Working directory: /Users/aleighamattison/Documents/hcde530/Mini project 1
Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *(What is it? Where did it come from? Paste the URL or citation from your MP1a submission.)* 

This is the Zillow Housing data set which is compiled by Zillow. There are different datasets available coverinng things like home listing, rental data, indices and forecasts.
URL: https://www.zillow.com/research/data/

**Why this dataset:** *(One sentence connecting it to your HCD work or research interests.)*
This is a valuable dataset and analysis for the HCD field as housing is an inherently human-centered problem space, as everyone either rents or buys a place to live, making insights into it essential.

**Three analytical questions:**

1. *(Question 1 from MP1a)* Are there seasonal patterns in listing counts, sales counts and prices for houses in the Seattle metro area that could inform my purchase timing? 
2. *(Question 2 from MP1a)* Over the last ten years, by what percentage has ZORI increased in my neighborhood, and how does that compare to the metro area and national ZORI growth? What does this show about the market?
3. *(Question 3 from MP1a)* How has ZORDI changed in my neighborhood over the last ten years, and how does that pattern compare to ZORI over the same period? What does this show about the market?

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*
A prospective buyer or renter could use this analysis when evaluating the housing market for buying or renting decisions. 

## Section 1.1 — Process Overview

I began by reviewing my datasets in depth to decide what I could visualize. I soon learned that I could not get a 2–3 bedroom breakdown for every metric, as I had assumed at the outset. For home values I could still work at the zipcode level, but condos were not always reported separately, so I sometimes accepted combined categories and adjusted my research questions. As I kept exploring, I found that several datasets lack zip-level detail like listings and sales, for example, are available only at the metro level—a constraint because I wanted the geography closest to my neighborhood.

To study **seasonality** in home values, I needed the raw monthly series export, the unfiltered month-to-month changes in estimated resale value. The smoothed alternative is essentially a moving average that dampens short-term movement and washes out seasonal patterns, so it would not have supported a seasonality-focused analysis.

After each dataframe load, I ran a short data-quality check (missing values, inconsistent strings, and related issues). When I synced my work, the CSV files were too large to commit, so I reverted those commits and staged only my notebook and scripts—not the raw data. I then built the Section 3 pandas analyses and the written summary to align with my charts.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [20]:
df = pd.read_csv('Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')  # ← replace with your filename

print(df.shape)
df.head()


(16000, 325)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2000-01-31,...,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,NaN,...,360508.218151,359357.323120,358704.192945,357914.487019,357184.945458,356313.526325,355192.552396,353542.180707,351726.204961,349977.121803
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,75652.151659,...,258365.902246,257410.661725,256544.844566,255900.940533,254875.738564,253658.789290,252401.819238,251400.152344,250737.754086,250114.428263
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,63819.647072,...,192097.810874,190853.219492,189734.280749,188790.664897,188397.012546,188305.277242,188120.935653,188011.459343,188163.137927,188171.748680
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,141898.776205,...,425081.251358,426154.298979,428197.521766,430548.467054,432944.893336,435824.040355,439653.718978,443615.620797,447754.078489,450070.676268
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,64589.223296,...,180073.533662,179359.250123,178410.259505,177504.736300,176830.130319,176548.681661,176132.955270,175823.070474,175614.705412,175108.513906


### Data quality check for the zip-level 2-bedroom ZHVI dataset
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [21]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')


Loaded: Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv  shape: (16000, 325)
Data quality check
shape: (16000, 325)
duplicate cols: False
total missing values: 1097187
columns with missing values: 292
dtypes:
{dtype('float64'): 316, <StringDtype(storage='python', na_value=nan)>: 6, dtype('int64'): 3}
numeric outliers >3σ: 74524
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 325 columns and 16,000 rows 

- What does each column represent?
Region ID is a unique identifier for the region. Size Rank indicates the population ranking of the area. Region Name is the zip code. Region Type specifies that it is a zip code. State Name is the state abbreviation. City is the city. Metro indicates the metro area. County Name is the county. After that, each column is a monthly breakdown of values starting in the year 2000.

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
There are some missing values. However, most gaps are empty months before Zillow had enough data to publish a 2-bedroom home value for that zip—not random errors. Coverage improves over time; recent months are largely complete. Missing City/Metro on some rows is incomplete labeling, not missing price history. For recent or well-covered zips, the series is still usable for analysis.

- Which column or columns will your analysis focus on, and why?
I will use the zip code column to focus the areas I would like to evaluate and the date columns to get values within the last 10 years. 

In [8]:
df = pd.read_csv('listings.csv')  # ← replace with your filename

print(df.shape)
df.head()

(928, 102)


,RegionID,SizeRank,RegionName,RegionType,StateName,2018-03-31,2018-04-30,2018-05-31,2018-06-30,2018-07-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,1421529.0,1500195.0,1592417.0,1660618.0,1709146.0,...,1329558.0,1373264.0,1380205.0,1373116.0,1362806.0,1322706.0,1239517.0,1158763.0,1115629.0,1155079.0
1,394913,1,"New York, NY",msa,NY,73707.0,80345.0,85864.0,90067.0,91881.0,...,47708.0,49038.0,48097.0,47265.0,46782.0,45632.0,41945.0,38103.0,35740.0,37158.0
2,753899,2,"Los Angeles, CA",msa,CA,21998.0,23784.0,25605.0,27109.0,28811.0,...,25128.0,26379.0,26688.0,26539.0,25801.0,24240.0,21853.0,20066.0,19707.0,21131.0
3,394463,3,"Chicago, IL",msa,IL,38581.0,42253.0,45757.0,47492.0,48984.0,...,23133.0,24026.0,24129.0,24076.0,23989.0,22985.0,20628.0,18354.0,17282.0,18258.0
4,394514,4,"Dallas, TX",msa,TX,24042.0,25876.0,28224.0,30490.0,32408.0,...,37501.0,38954.0,39091.0,38265.0,37323.0,35777.0,33494.0,31310.0,30185.0,31238.0


### Data quality check for listings.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [22]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='listings.csv')


Loaded: listings.csv  shape: (928, 102)
Data quality check
shape: (928, 102)
duplicate cols: False
total missing values: 1192
columns with missing values: 62
dtypes:
{dtype('float64'): 97, <StringDtype(storage='python', na_value=nan)>: 3, dtype('int64'): 2}
numeric outliers >3σ: 173
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 928 rows and 102 columns 

- What does each column represent?
Region ID is a unique identifier for the region. Size Rank indicates the population ranking of the area. Region Name is the city and state or a national category (USA). Region Type specifies msa which is metropolitan statistical area. After that, each column is a monthly count of listing for sale since the year 2018 which explains the slight gap in one of my line charts. 

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
There are many numeric outliers above 3 sigma and also some missing values

- Which column or columns will your analysis focus on, and why?
I will use the RegionName column to focus the areas I would like to evaluate and the date columns to review numbers of listings posted within the last 8 years.

In [10]:
df = pd.read_csv('sales.csv')  # ← replace with your filename

print(df.shape)
df.head()

(301, 223)


,RegionID,SizeRank,RegionName,RegionType,StateName,2008-02-29,2008-03-31,2008-04-30,2008-05-31,2008-06-30,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,203822.0,235967.0,261747.0,288767.0,301790.0,...,359536.0,357359.0,341460.0,326838.0,332061.0,263951.0,303764.0,217693.0,235352.0,305061.0
1,394913,1,"New York, NY",msa,NY,8526.0,9055.0,10043.0,10470.0,11319.0,...,13598.0,14854.0,14364.0,13799.0,13639.0,11015.0,13184.0,10394.0,8466.0,10271.0
2,753899,2,"Los Angeles, CA",msa,CA,4135.0,5052.0,6078.0,6865.0,7220.0,...,6622.0,6985.0,6499.0,6645.0,7009.0,5395.0,6135.0,4432.0,5017.0,6589.0
3,394463,3,"Chicago, IL",msa,IL,5601.0,6961.0,7327.0,7992.0,8824.0,...,10412.0,10147.0,9277.0,8541.0,8770.0,6581.0,7432.0,5188.0,5513.0,8159.0
4,394514,4,"Dallas, TX",msa,TX,4918.0,5588.0,6030.0,6727.0,6720.0,...,7071.0,7148.0,6585.0,6063.0,6044.0,4749.0,5751.0,4113.0,4967.0,6550.0


### Data quality check for sales.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [23]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='sales.csv')


Loaded: sales.csv  shape: (301, 223)
Data quality check
shape: (301, 223)
duplicate cols: False
total missing values: 361
columns with missing values: 165
dtypes:
{dtype('float64'): 218, <StringDtype(storage='python', na_value=nan)>: 3, dtype('int64'): 2}
numeric outliers >3σ: 232
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 301 rows and 223 columns 

- What does each column represent?
These are repeats of the same columns except the monthly columns here go back to 2008 and are counts of the total sales.

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
There are many outliers (3sigma test) and some missing values as well

- Which column or columns will your analysis focus on, and why?
I will use the RegionName column to focus the areas I would like to evaluate and the date columns to review numbers of listings sold within the last 10 years.

In [2]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_csv('Metro_median_sale_price_now_uc_sfrcondo_month.csv')  # ← replace with your filename

print(df.shape)
df.head()

(301, 223)


,RegionID,SizeRank,RegionName,RegionType,StateName,2008-02-29,2008-03-31,2008-04-30,2008-05-31,2008-06-30,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,170250.0,175000.0,177000.0,180000.0,185000.0,...,375000.0,370000.0,368000.0,360000.0,362500.0,360000.0,355000.0,350000.0,357000.0,369494.0
1,394913,1,"New York, NY",msa,NY,400000.0,390000.0,390000.0,392000.0,400000.0,...,685000.0,686000.0,685000.0,660000.0,654250.0,665000.0,650000.0,655000.0,650000.0,646468.0
2,753899,2,"Los Angeles, CA",msa,CA,470000.0,455000.0,457750.0,440000.0,435000.0,...,1000000.0,960000.0,949000.0,955000.0,945000.0,945000.0,923000.0,920000.0,950000.0,974029.0
3,394463,3,"Chicago, IL",msa,IL,218750.0,220000.0,223000.0,229000.0,235000.0,...,350000.0,340000.0,340000.0,325000.0,325000.0,324000.0,315000.0,314000.0,320000.0,331995.0
4,394514,4,"Dallas, TX",msa,TX,138000.0,145500.0,145000.0,150000.0,156000.0,...,408000.0,400000.0,387000.0,380000.0,380000.0,374100.0,370000.0,370000.0,379500.0,390123.0


### Data quality check for Metro_median_sale_price_now_uc_sfrcondo_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [24]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Metro_median_sale_price_now_uc_sfrcondo_month.csv')


Loaded: Metro_median_sale_price_now_uc_sfrcondo_month.csv  shape: (301, 223)
Data quality check
shape: (301, 223)
duplicate cols: False
total missing values: 333
columns with missing values: 159
dtypes:
{dtype('float64'): 218, <StringDtype(storage='python', na_value=nan)>: 3, dtype('int64'): 2}
numeric outliers >3σ: 1621
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 301 rows and 223 columns 

- What does each column represent?
These are repeats of the same columns except the monthly columns here go back to 2008 and are the median value sold listing.

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
Again, lots of 3sigma outliers

- Which column or columns will your analysis focus on, and why?
I will use the RegionName column to focus the areas I would like to evaluate and the date columns to pull in the value of the properties sold. 

In [14]:
df = pd.read_csv('Zori_zip_uc_sfrcondomfr_sm_month.csv')  # ← A smoothed measure of the typical observed market rate rent across a given region. ZORI is a repeat-rent index that is weighted to the rental housing stock to ensure representativeness across the entire market, not just those homes currently listed for-rent. 
print(df.shape)
df.head()

(8188, 144)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2015-01-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,1396.420611,...,1752.349869,1761.399334,1768.181320,1762.748802,1752.914282,1739.839933,1735.345787,1716.202697,1704.095737,1683.654847
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2305.088360,2379.926927,2310.000000
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,1238.667257,...,1825.135558,1823.161847,1814.078021,1805.419803,1804.616865,1793.828474,1804.151902,1793.823253,1797.458821,1779.454743
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2700.000000
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,1093.863007,...,1576.421608,1573.457268,1579.511547,1579.319611,1574.482724,1564.271077,1559.644039,1559.997423,1554.930010,1544.184291


### Data quality check for Zori_zip_uc_sfrcondomfr_sm_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [25]:
# Data quality check — run Setup cell once first
data_quality_check(csv_path='Zori_zip_uc_sfrcondomfr_sm_month.csv')


Loaded: Zori_zip_uc_sfrcondomfr_sm_month.csv  shape: (8188, 144)
Data quality check
shape: (8188, 144)
duplicate cols: False
total missing values: 677111
columns with missing values: 137
dtypes:
{dtype('float64'): 135, <StringDtype(storage='python', na_value=nan)>: 6, dtype('int64'): 3}
numeric outliers >3σ: 2543
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 8188 rows and 144 columns 

- What does each column represent?
These are repeats of the same columns except the value is the Zori from zillow which is a a smoothed measure of the typical observed market rate rent across a given region

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
Missing values are expected at zip level: Zillow only reports ZORI where there’s enough rental data. Many zips have leading blank months before their series starts. For zip 98004 and metro/national series used in my charts (2016+), coverage is sufficient; I’m not treating these as data errors.

- Which column or columns will your analysis focus on, and why?
I will use the RegionName column to focus the areas I would like to evaluate and the date columns to pull in the Zori value in order to view its trends and growth rates. 

In [41]:
df = pd.read_csv('Zori_Metro_uc_sfrcondomfr_sm_month.csv')  # ← A smoothed measure of the typical observed market rate rent across a given region. ZORI is a repeat-rent index that is weighted to the rental housing stock to ensure representativeness across the entire market, not just those homes currently listed for-rent. 
print(df.shape)
df.head()

(719, 140)


,RegionID,SizeRank,RegionName,RegionType,StateName,2015-01-31,2015-02-28,2015-03-31,2015-04-30,2015-05-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,1136.747272,1143.002520,1151.566003,1160.295331,1168.904681,...,1901.444662,1905.091763,1906.236631,1904.921138,1901.310487,1895.636662,1891.140259,1892.439923,1899.581978,1910.424994
1,394913,1,"New York, NY",msa,NY,2184.191044,2198.930168,2217.824385,2236.403118,2250.771613,...,3290.539695,3320.703116,3341.252886,3338.784221,3326.153715,3305.460325,3293.370385,3290.776879,3308.005112,3337.139888
2,753899,2,"Los Angeles, CA",msa,CA,1739.529369,1750.819968,1765.883551,1780.754656,1795.593017,...,2884.341555,2888.500950,2890.075156,2889.810116,2886.795063,2879.898972,2870.937038,2873.470854,2880.684896,2894.901157
3,394463,3,"Chicago, IL",msa,IL,1374.256131,1381.587911,1391.546391,1401.007023,1410.826622,...,2126.664004,2137.955170,2142.052174,2140.203901,2134.356115,2129.121344,2127.097179,2138.409168,2157.030613,2179.817593
4,394514,4,"Dallas, TX",msa,TX,1049.220308,1053.961615,1061.339334,1072.453106,1081.657882,...,1665.128879,1663.359463,1659.929336,1654.747658,1648.248483,1641.725612,1635.817921,1633.592707,1637.284321,1644.537859


### Data quality check for Zori_Metro_uc_sfrcondomfr_sm_month.csv
The following cell verifies missing values, duplicate columns, data types, and string formatting issues for this dataset.


In [ ]:
# Data quality check — run Setup cell once first
data_quality_check(y csv_path='Zori_Metro_uc_sfrcondomfr_sm_month.csv')


Loaded: Zori_Metro_uc_sfrcondomfr_sm_month.csv  shape: (719, 140)
Data quality check
shape: (719, 140)
duplicate cols: False
total missing values: 47030
columns with missing values: 135
dtypes:
{dtype('float64'): 135, <StringDtype(storage='python', na_value=nan)>: 3, dtype('int64'): 2}
numeric outliers >3σ: 1098
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 719 rows and 140 columns 

- What does each column represent?
This is a repeat of the above excepy by metro areas so I could pull in data for the national average 

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
Almost half of the values are missing...

- Which column or columns will your analysis focus on, and why?
I will use the RegionName column to focus the Zori national average.

In [32]:
df = pd.read_csv('Zordi_Metro_uc_condo_month.csv')  # ← replace with your filename

print(df.shape)
df.head()

(754, 75)


,RegionID,SizeRank,RegionName,RegionType,StateName,2020-06-30,2020-07-31,2020-08-31,2020-09-30,2020-10-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,57.0,54.0,47.0,43.0,43.0,...,37.0,32.0,25.0,25.0,29.0,33.0,36.0,40.0,43.0,45.0
1,394913,1,"New York, NY",msa,NY,60.0,48.0,37.0,35.0,43.0,...,88.0,78.0,64.0,65.0,72.0,79.0,84.0,89.0,92.0,95.0
2,753899,2,"Los Angeles, CA",msa,CA,96.0,99.0,89.0,74.0,66.0,...,57.0,52.0,45.0,41.0,37.0,38.0,45.0,50.0,51.0,48.0
3,394463,3,"Chicago, IL",msa,IL,72.0,63.0,47.0,35.0,34.0,...,104.0,83.0,62.0,50.0,47.0,48.0,56.0,69.0,84.0,92.0
4,394514,4,"Dallas, TX",msa,TX,65.0,67.0,60.0,48.0,39.0,...,18.0,15.0,12.0,10.0,9.0,9.0,10.0,12.0,14.0,14.0


In [46]:
data_quality_check( csv_path='Zordi_Metro_uc_condo_month.csv')


Loaded: Zordi_Metro_uc_condo_month.csv  shape: (754, 75)
Data quality check
shape: (754, 75)
duplicate cols: False
total missing values: 36593
columns with missing values: 71
dtypes:
{dtype('float64'): 70, <StringDtype(storage='python', na_value=nan)>: 3, dtype('int64'): 2}
numeric outliers >3σ: 317
string formatting issues: none


- How many rows and columns does your dataset have? 
It has 754 rows and 75 columns 

- What does each column represent?
This is a repeat of the above excepy the value is the Zordi which is a measure of the typical observed rental market engagement across a region. ZORDI tracks engagement on Zillow’s rental listings to proxy changes in rental demand.

- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
significant amount of missing data

- Which column or columns will your analysis focus on, and why?
I will use the RegionName column to focus the Zordi within seattle to evalute demand vs rent trends. 

In [26]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


In [27]:
# Summary statistics for numeric columns
df.describe()

,id,rating,helpful_votes
count,500.000000,500.000000,500.000000
mean,250.500000,3.946000,23.464000
std,144.481833,1.184013,13.766471
min,1.000000,1.000000,0.000000
25%,125.750000,3.000000,11.000000
50%,250.500000,4.000000,23.500000
75%,375.250000,5.000000,35.000000
max,500.000000,5.000000,47.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** Are there seasonal patterns in listins counts, sales counts and price for houses in the Seattle metro area that could inform my purchase timing? 

In [33]:
# Question 1 — Seasonal patterns in Seattle listings, sales, and median price (2016+)
import pandas as pd

START_DATE = '2016-01-01'
SEATTLE = 'Seattle, WA'
MONTH_LABELS = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
                7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}


def seattle_series(path, value_name='value'):
    """Monthly series for Seattle MSA from a wide Zillow CSV."""
    df = pd.read_csv(path)
    date_cols = [c for c in df.columns if c.startswith('20') and '-' in c]
    row = df[(df['RegionName'] == SEATTLE) & (df['RegionType'] == 'msa')].iloc[0]
    s = pd.to_numeric(row[date_cols], errors='coerce')
    s.index = pd.to_datetime(date_cols)
    s = s[s.index >= START_DATE]
    s.name = value_name
    return s


def monthly_profile(s):
    """Average by calendar month and percent vs overall mean."""
    df = s.dropna().to_frame('value')
    df['month'] = df.index.month
    profile = df.groupby('month')['value'].agg(['mean', 'median', 'std', 'count'])
    profile['pct_vs_avg'] = (profile['mean'] / profile['mean'].mean() - 1) * 100
    profile.index = [MONTH_LABELS[m] for m in profile.index]
    return profile.round(1)


listings = seattle_series('listings.csv', 'Listings')
sales = seattle_series('sales.csv', 'Sales')
price = seattle_series('Metro_median_sale_price_now_uc_sfrcondo_month.csv', 'MedianPrice')

print('=== 1. Seattle monthly profile (% vs annual average) ===\n')
for label, s in [('Listings', listings), ('Sales', sales), ('Median price', price)]:
    print(f'--- {label} ---')
    print(monthly_profile(s)[['mean', 'pct_vs_avg']])
    print()


=== 1. Seattle monthly profile (% vs annual average) ===

--- Listings ---
        mean  pct_vs_avg
Jan   6624.8       -26.5
Feb   6026.8       -33.2
Mar   6813.9       -24.4
Apr   7510.5       -16.7
May   8929.5        -1.0
Jun  10024.5        11.2
Jul  10959.2        21.6
Aug  11266.0        25.0
Sep  11257.5        24.9
Oct  10882.9        20.7
Nov   9823.5         9.0
Dec   8068.4       -10.5

--- Sales ---
       mean  pct_vs_avg
Jan  2914.5       -39.4
Feb  3334.5       -30.6
Mar  4550.7        -5.3
Apr  5036.0         4.8
May  5589.3        16.3
Jun  6104.9        27.0
Jul  5804.6        20.7
Aug  5747.6        19.6
Sep  5186.7         7.9
Oct  5125.3         6.6
Nov  4319.5       -10.2
Dec  3976.1       -17.3

--- Median price ---
         mean  pct_vs_avg
Jan  505189.5        -8.7
Feb  529722.7        -4.2
Mar  557290.3         0.7
Apr  562747.4         1.7
May  573413.8         3.7
Jun  577168.7         4.3
Jul  568832.5         2.8
Aug  562469.1         1.7
Sep  554941.7    

In [34]:
# 2. Seattle vs national seasonal pattern (same method as listings/sales chart)
def national_monthly_mean(path):
  df = pd.read_csv(path)
  date_cols = [c for c in df.columns if c.startswith('20') and '-' in c]
  long = df[df['RegionType'] != 'country'].melt(
      value_vars=date_cols, var_name='Date', value_name='value'
  )
  long['Date'] = pd.to_datetime(long['Date'])
  long['value'] = pd.to_numeric(long['value'], errors='coerce')
  long = long[long['Date'] >= START_DATE]
  nat = long.groupby('Date')['value'].mean()
  out = nat.dropna().to_frame('value')
  out['month'] = out.index.month
  return out.groupby('month')['value'].mean()


def compare_seattle_national(sea_series, path, label):
  sea_by_month = sea_series.dropna().to_frame('value')
  sea_by_month['month'] = sea_by_month.index.month
  sea_avg = sea_by_month.groupby('month')['value'].mean()
  nat_avg = national_monthly_mean(path)
  compare = pd.DataFrame({'Seattle': sea_avg, 'National': nat_avg})
  compare.index = [MONTH_LABELS[m] for m in compare.index]
  compare['Seattle_pct'] = (compare['Seattle'] / compare['Seattle'].mean() - 1) * 100
  compare['National_pct'] = (compare['National'] / compare['National'].mean() - 1) * 100
  compare['Seattle_minus_National_pp'] = (
      compare['Seattle_pct'] - compare['National_pct']
  ).round(1)
  print(f'=== {label}: Seattle vs national (% vs each series mean) ===')
  print(compare.round(1))
  print()


compare_seattle_national(listings, 'listings.csv', 'Listings')
compare_seattle_national(sales, 'sales.csv', 'Sales')

# 4. Seasonal swing summary (peak month, low month, size of swing)
def seasonal_summary(s, label):
  p = monthly_profile(s)
  return {
      'metric': label,
      'peak_month': p['pct_vs_avg'].idxmax(),
      'peak_pct': p['pct_vs_avg'].max(),
      'low_month': p['pct_vs_avg'].idxmin(),
      'low_pct': p['pct_vs_avg'].min(),
      'swing_pp': (p['pct_vs_avg'].max() - p['pct_vs_avg'].min()).round(1),
  }


summary = pd.DataFrame([
    seasonal_summary(listings, 'Listings'),
    seasonal_summary(sales, 'Sales'),
    seasonal_summary(price, 'Median price'),
])
print('=== 4. Seasonal swing summary ===')
print(summary.to_string(index=False))

=== Listings: Seattle vs national (% vs each series mean) ===
     Seattle  National  Seattle_pct  National_pct  Seattle_minus_National_pp
Jan   6624.8    1081.2        -26.5         -10.6                      -15.9
Feb   6026.8    1035.5        -33.2         -14.4                      -18.8
Mar   6813.9    1106.8        -24.4          -8.5                      -16.0
Apr   7510.5    1142.4        -16.7          -5.5                      -11.2
May   8929.5    1211.4         -1.0           0.2                       -1.1
Jun  10024.5    1264.1         11.2           4.5                        6.6
Jul  10959.2    1308.1         21.6           8.2                       13.4
Aug  11266.0    1326.5         25.0           9.7                       15.3
Sep  11257.5    1319.5         24.9           9.1                       15.7
Oct  10882.9    1302.8         20.7           7.7                       13.0
Nov   9823.5    1250.6          9.0           3.4                        5.5
Dec   8068.4  

In [35]:
# 3. Month × year table — is the pattern repeated across years?
def month_year_pivot(s, label):
  df = s.dropna().to_frame('value')
  df['year'] = df.index.year
  df['month'] = df.index.month
  pivot = df.pivot_table(index='month', columns='year', values='value', aggfunc='mean')
  pivot.index = [MONTH_LABELS[m] for m in pivot.index]
  print(f'=== 3. {label}: average by month and year ===')
  print(pivot.round(0))
  print()


month_year_pivot(listings, 'Listings')

# 5. Correlation between listings and sales (all months + monthly averages)
sea = pd.DataFrame({'listings': listings, 'sales': sales}).dropna()
by_month = sea.groupby(sea.index.month)[['listings', 'sales']].mean()
print('=== 5. Listings vs sales correlation ===')
print('Across all months in sample:')
print(sea[['listings', 'sales']].corr().round(3))
print('\nAcross 12 monthly averages:')
print(by_month.corr().round(3))

=== 3. Listings: average by month and year ===
year     2018     2019     2020     2021     2022    2023     2024     2025  \
Jan       NaN   9544.0   6431.0   6372.0   4513.0  6781.0   5103.0   6393.0   
Feb       NaN   8300.0   5757.0   5917.0   4223.0  5740.0   4786.0   6010.0   
Mar    7287.0   8772.0   6529.0   6792.0   5492.0  5715.0   5352.0   6862.0   
Apr    8277.0   9585.0   7293.0   7831.0   6947.0  5894.0   6282.0   7975.0   
May   10148.0  11590.0   8275.0   8998.0   8548.0  6520.0   7567.0   9790.0   
Jun   11666.0  13026.0   8830.0   9625.0   9917.0  7006.0   8793.0  11333.0   
Jul   12963.0  13915.0   9744.0  10050.0  11295.0  7470.0   9729.0  12508.0   
Aug   13567.0  13639.0  10453.0  10143.0  11941.0  7698.0  10064.0  12623.0   
Sep   13986.0  12970.0  10836.0   9966.0  11850.0  7774.0  10169.0  12509.0   
Oct   14072.0  12071.0  10655.0   9357.0  11111.0  7656.0  10059.0  12082.0   
Nov   13226.0  10489.0   9447.0   8017.0   9913.0  7122.0   9231.0  11143.0   
Dec  

**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

Using my Seattle vs national listings/sales chart (2016–2026) and the above analysis, Seattle shows strong seasonality: listings peak in late summer (~25% above average in August) and bottom in winter (~33% below in February); sales peak in June–July and are lowest in January. The national series follow the same pattern but with smaller swings, so Seattle’s market is more seasonal than the U.S. average on this measure. My median sale price chart shows a weaker seasonal cycle: prices are somewhat higher in late spring and lower in winter (~9% below peak). For purchase timing, the charts suggest winter may offer less competition and slightly lower typical prices, while late spring–summer offers more inventory but more competition and somewhat higher medians so there are tradeoffs rather than a single “best” month.

I expected seasonality in the listings and sales as this is a similar pattern for renters. However, I thought there might be more of a pattern for the price. From here, I would likely investigate other compareable cities and if their price movements have any seasonality to them to help benchmark my reseults. 

**Question 2:** *Over the last ten years, by what percentage has ZORI increased in my neighborhood, and how does that compare to the metro area and national ZORI growth? What does this show about the market?*

In [36]:
# Question 2 analysis — Bellevue (98004) vs Seattle metro vs U.S. (2016–2026)
import pandas as pd

START_DATE = '2016-01-01'
END_DATE = '2026-12-31'

zori_zip = pd.read_csv('Zori_zip_uc_sfrcondomfr_sm_month.csv')
zori_metro = pd.read_csv('Zori_Metro_uc_sfrcondomfr_sm_month.csv')


def zori_series(df, region, region_type):
    date_cols = [c for c in df.columns if c.startswith('20')]
    row = df[
        (df['RegionName'].astype(str) == str(region))
        & (df['RegionType'] == region_type)
    ].iloc[0]
    s = pd.to_numeric(row[date_cols], errors='coerce')
    s.index = pd.to_datetime(date_cols)
    return s[(s.index >= START_DATE) & (s.index <= END_DATE)].dropna()


series = {
    'Bellevue (98004)': zori_series(zori_zip, '98004', 'zip'),
    'Seattle Metro': zori_series(zori_metro, 'Seattle, WA', 'msa'),
    'United States': zori_series(zori_metro, 'United States', 'country'),
}

# 1) Total growth: first month to last month
print('=== Total ZORI growth ===')
rows = []
for label, s in series.items():
    start_val, end_val = s.iloc[0], s.iloc[-1]
    rows.append({
        'Region': label,
        'Start': s.index[0].strftime('%Y-%m'),
        'End': s.index[-1].strftime('%Y-%m'),
        'Start ZORI ($)': round(start_val, 0),
        'End ZORI ($)': round(end_val, 0),
        'Total growth (%)': round((end_val / start_val - 1) * 100, 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

# 2) Annual year-over-year growth (matches Section 4 bar chart)
print('\n=== Annual YoY ZORI growth (%) ===')
yoy_frames = []
for label, s in series.items():
    annual = s.groupby(s.index.year).last()
    yoy = (annual.pct_change() * 100).round(2)
    yoy_frames.append(pd.DataFrame({'Year': yoy.index, label: yoy.values}))
yoy_table = yoy_frames[0]
for frame in yoy_frames[1:]:
    yoy_table = yoy_table.merge(frame, on='Year', how='outer')
print(yoy_table.dropna(how='all', subset=[c for c in yoy_table.columns if c != 'Year']).to_string(index=False))


=== Total ZORI growth ===
          Region   Start     End  Start ZORI ($)  End ZORI ($)  Total growth (%)
Bellevue (98004) 2016-05 2026-03          2070.0        2842.0              37.3
   Seattle Metro 2016-01 2026-03          1408.0        2192.0              55.7
   United States 2016-01 2026-03          1191.0        1910.0              60.4

=== Annual YoY ZORI growth (%) ===
 Year  Bellevue (98004)  Seattle Metro  United States
 2017              3.43           6.39           4.38
 2018              1.81           3.84           4.26
 2019              5.16           4.66           3.96
 2020             -6.52          -3.54           1.80
 2021             14.80          14.98          15.33
 2022              2.72           4.52           7.08
 2023              2.70           2.07           2.96
 2024              5.05           3.37           2.93
 2025             -0.92           2.58           2.12
 2026              4.19           0.58           1.02


**Interpretation:**  
From 2016 through early 2026, ZORI in Bellevue (98004) increased by about 37% (from roughly $2,070 to $2,842 per month), while Seattle metro rose about 56% and the United States about 60%. My neighborhood’s rent index grew, but more slowly than both the metro and national benchmarks in this window.

That pattern suggests Bellevue’s observed market rent growth was moderate relative to the broader Seattle area and the country—not a rent decline, but not the fastest-appreciating market in the comparison. The line chart shows all three series trending upward, with sharper national and metro gains especially after 2020 likely due to the COVID-19 Pandemic. 

What this shows about the market: Rising ZORI at zip, metro, and national levels points to sustained rental market pressure over the decade. The gap between 98004 and Seattle/U.S. may reflect local supply–demand balance, housing stock, or when Zillow’s zip-level series starts (May 2016 for 98004). I would investigate nearby zips and rent burden next to understand affordability for residents.

**Question 3:** *How has ZORDI changed in my neighborhood over the last six years, and how does that pattern compare to ZORI over the same period? What does this show about the market?*

*Note: ZORDI is only available at metro level here, so this analysis uses Seattle metro ZORI vs ZORDI from June 2020, when both series begin to update my original research question.*

In [37]:
# Question 3 analysis — Seattle metro ZORI vs ZORDI (from 2020-06, when both start)
import pandas as pd

START_DATE = '2020-06-01'
END_DATE = '2026-12-31'
SEATTLE = 'Seattle, WA'

zori_metro = pd.read_csv('Zori_Metro_uc_sfrcondomfr_sm_month.csv')
zordi_metro = pd.read_csv('Zordi_Metro_uc_condo_month.csv')


def metro_series(df, region, region_type='msa'):
    date_cols = [c for c in df.columns if c.startswith('20')]
    row = df[(df['RegionName'] == region) & (df['RegionType'] == region_type)].iloc[0]
    s = pd.to_numeric(row[date_cols], errors='coerce')
    s.index = pd.to_datetime(date_cols)
    return s[(s.index >= START_DATE) & (s.index <= END_DATE)].dropna()


zori = metro_series(zori_metro, SEATTLE)
zordi = metro_series(zordi_metro, SEATTLE)

# 1) Total growth for each index
print('=== Total growth (June 2020 → latest month) ===')
for label, s in [('ZORI (rent index)', zori), ('ZORDI (demand index)', zordi)]:
    start_val, end_val = s.iloc[0], s.iloc[-1]
    print(
        f"{label}: {round((end_val / start_val - 1) * 100, 1)}%  "
        f"({s.index[0].strftime('%Y-%m')} ${start_val:,.0f} → "
        f"{s.index[-1].strftime('%Y-%m')} ${end_val:,.0f})"
    )

# 2) Side-by-side monthly comparison
compare = pd.DataFrame({'ZORI': zori, 'ZORDI': zordi}).dropna()
compare['ZORI_pct_change'] = (compare['ZORI'] / compare['ZORI'].iloc[0] - 1) * 100
compare['ZORDI_pct_change'] = (compare['ZORDI'] / compare['ZORDI'].iloc[0] - 1) * 100

print('\n=== Latest indexed change from June 2020 (= 0%) ===')
print(compare[['ZORI_pct_change', 'ZORDI_pct_change']].tail(3).round(1))

print('\n=== Correlation (same months) ===')
print(compare[['ZORI', 'ZORDI']].corr().round(3))

=== Total growth (June 2020 → latest month) ===
ZORI (rent index): 23.9%  (2020-06 $1,769 → 2026-03 $2,192)
ZORDI (demand index): -48.5%  (2020-06 $66 → 2026-03 $34)

=== Latest indexed change from June 2020 (= 0%) ===
            ZORI_pct_change  ZORDI_pct_change
2026-01-31             23.1             -54.5
2026-02-28             23.4             -48.5
2026-03-31             23.9             -48.5

=== Correlation (same months) ===
        ZORI  ZORDI
ZORI   1.000 -0.101
ZORDI -0.101  1.000


**Interpretation:**  
From **June 2020** through early **2026**, **Seattle metro ZORI rose about 24%** (typical observed rent increased), while **ZORDI fell about 49%** (rental listing engagement on Zillow declined). The two series move in **opposite directions** over this period: rents trended up, but measured renter **demand/engagement** trended down.

That divergence is what the Section 4 chart shows — ZORI climbing on the left axis and ZORDI falling on the right. This does **not** mean Seattle had no rental demand at all; ZORDI is a **condo-focused engagement index**, and it only starts in mid-2020 for Seattle, so I compare from the shared start date rather than the full 2016 window.

**What this shows about the market:** Rising ZORI with falling ZORDI suggests rent levels increased even as listing engagement softened consistent with a tighter or more expensive rental market where fewer listings get the same level of renter activity. Price pressure (ZORI) and rental demand (ZORDI) tell different stories, so it helps to look at both and what they could indicate together. 



---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [23]:
fig.update_layout(
    title_text="ZORI Growth Analysis: 98004 Zip Code vs Seattle Metro vs National Average (2016-2026)",
    height=500,
    width=1400,
    showlegend=True,
    hovermode='x unified',
    barmode='group',
    legend=dict(
        x=1.05,  # Position legend outside the chart area to the right
        y=0.5,
        xanchor='left',
        yanchor='middle'
    )
)

**ZORI Growth Analysis (supports Question 2):**  
This visualization answers **Question 2** — *Over the last ten years, by what percentage has ZORI increased in my neighborhood, and how does that compare to the metro area and national ZORI growth?*

**Why this chart type:** I split the view into two panels because they answer different parts of the question. The **line chart (left)** shows monthly ZORI levels from 2016–2026 for **Bellevue (98004)**, **Seattle metro**, and the **United States**. The right format for a time series when you want to see *direction, timing, and relative level* over a decade. The **grouped bar chart (right)** shows **year-over-year percent change** by calendar year, which makes it easier to compare *how fast* rent grew in each region in a given year — not just where each series ended up.

**What I want the reader to take away:** All three regions saw **rising ZORI** over this period, but **Bellevue (98004) grew more slowly (+37%)** than **Seattle metro (+56%)** and the **U.S. (+60%)** (beginning of the series to the end)— matching the pandas totals in Section 3. The lines separate over time, especially after 2020, while the bars show that growth was **not steady every year** (for example, a weaker year around 2020 and a stronger spike in 2021 for the zip). Together, the charts support the Section 3 finding that Bellevue’s rent index increased, but **lagged broader metro and national rent growth** — useful context for how “hot” the local rental market has been relative to benchmarks.

In [24]:
# ZORI vs ZORDI Comparison for Seattle Metro Area (2016-2026)
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load ZORI and ZORDI data
zori_metro = pd.read_csv('Zori_Metro_uc_sfrcondomfr_sm_month.csv')
zordi_metro = pd.read_csv('Zordi_Metro_uc_condo_month.csv')

# Filter for Seattle metro area
seattle_zori = zori_metro[zori_metro['RegionName'] == 'Seattle, WA'].copy()
seattle_zordi = zordi_metro[zordi_metro['RegionName'] == 'Seattle, WA'].copy()

# Get date columns (starting from 2016)
zori_date_cols = [col for col in seattle_zori.columns if col.startswith('20') and col >= '2016-01-31']
zordi_date_cols = [col for col in seattle_zordi.columns if col.startswith('20') and col >= '2016-01-31']

# Melt ZORI data
zori_long = seattle_zori.melt(
    id_vars=['RegionID', 'RegionName', 'RegionType', 'StateName'],
    value_vars=zori_date_cols,
    var_name='Date',
    value_name='ZORI'
)

# Melt ZORDI data
zordi_long = seattle_zordi.melt(
    id_vars=['RegionID', 'RegionName', 'RegionType', 'StateName'],
    value_vars=zordi_date_cols,
    var_name='Date',
    value_name='ZORDI'
)

# Convert dates and merge
zori_long['Date'] = pd.to_datetime(zori_long['Date'])
zordi_long['Date'] = pd.to_datetime(zordi_long['Date'])

# Merge the datasets
merged = pd.merge(zori_long[['Date', 'ZORI']], zordi_long[['Date', 'ZORDI']], on='Date', how='inner')
merged = merged.sort_values('Date')

# Create dual-axis subplot
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add ZORI trace (left axis - rent prices)
fig.add_trace(
    go.Scatter(
        x=merged['Date'], 
        y=merged['ZORI'], 
        name="ZORI (Rent Index)",
        line=dict(color='blue', width=2)
    ),
    secondary_y=False,
)

# Add ZORDI trace (right axis - demand index)
fig.add_trace(
    go.Scatter(
        x=merged['Date'], 
        y=merged['ZORDI'], 
        name="ZORDI (Demand Index)",
        line=dict(color='red', width=2)
    ),
    secondary_y=True,
)

# Update layout
fig.update_layout(
    title_text="ZORI vs ZORDI Trends in Seattle Metro Area (2016-2026)",
    height=500,
    width=1000,
    hovermode='x unified'
)

# Update axes labels
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="ZORI (Rent Index - $)", secondary_y=False)
fig.update_yaxes(title_text="ZORDI (Demand Index)", secondary_y=True)

# Show the chart
fig.show()

**ZORI vs ZORDI Trends (supports Question 3):**  
This visualization answers **Question 3** — *How has ZORDI changed over the last ten years, and how does that pattern compare to ZORI over the same period? What does this show about the market?* *(ZORDI is only available at metro level here, so both series are for **Seattle metro**; ZORDI begins in mid-2020.)*

**Why this chart type:** I used a **dual-axis line chart** because ZORI (typical rent, in dollars) and ZORDI (listing engagement index) are **different metrics on different scales**, but both are **monthly time series** that need to be compared over the same timeline. Putting ZORI on the left axis and ZORDI on the right lets the reader see **whether rent and demand move together or diverge** without forcing them onto one misleading scale. A line chart fits because the question is about **change over time**, not a single snapshot comparison.

**What I want the reader to take away:** From **June 2020** onward (when both series start), **ZORI rose about 24%** while **ZORDI fell about 49%** so the lines move in **opposite directions**. That supports the Section 3 finding that **rent levels increased even as rental listing engagement on Zillow declined**. For market interpretation, that gap suggests **price pressure without matching growth in renter activity**.

In [11]:
# Seattle vs national listings and sales comparison
import pandas as pd
import plotly.express as px

listings = pd.read_csv('listings.csv')
sales = pd.read_csv('sales.csv')

listings_date_cols = [c for c in listings.columns if c.startswith('20') and '-' in c]
sales_date_cols = [c for c in sales.columns if c.startswith('20') and '-' in c]

listings_long = listings.melt(
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'],
    value_vars=listings_date_cols,
    var_name='Date',
    value_name='Count'
)
sales_long = sales.melt(
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'],
    value_vars=sales_date_cols,
    var_name='Date',
    value_name='Count'
)

listings_long['Date'] = pd.to_datetime(listings_long['Date'])
sales_long['Date'] = pd.to_datetime(sales_long['Date'])

start_date = pd.Timestamp('2016-01-01')
listings_long = listings_long[listings_long['Date'] >= start_date]
sales_long = sales_long[sales_long['Date'] >= start_date]

seattle_listings = listings_long[
    (listings_long['RegionName'] == 'Seattle, WA') &
    (listings_long['RegionType'] == 'msa')
].copy()
seattle_sales = sales_long[
    (sales_long['RegionName'] == 'Seattle, WA') &
    (sales_long['RegionType'] == 'msa')
].copy()

national_listings = listings_long[listings_long['RegionType'] != 'country']
national_sales = sales_long[sales_long['RegionType'] != 'country']

national_listings = national_listings.groupby('Date')['Count'].mean().reset_index()
national_sales = national_sales.groupby('Date')['Count'].mean().reset_index()

seattle_listings['Series'] = 'Seattle Listings'
seattle_sales['Series'] = 'Seattle Sales'
national_listings['Series'] = 'National Avg Listings'
national_sales['Series'] = 'National Avg Sales'

plot_df = pd.concat([
    seattle_listings[['Date', 'Count', 'Series']],
    seattle_sales[['Date', 'Count', 'Series']],
    national_listings[['Date', 'Count', 'Series']],
    national_sales[['Date', 'Count', 'Series']]
], ignore_index=True)

fig = px.line(
    plot_df.sort_values(['Series', 'Date']),
    x='Date',
    y='Count',
    color='Series',
    title='Seattle Listings and Sales vs National Average (Last 10 Years)',
    labels={
        'Date': 'Date',
        'Count': 'Number of Homes',
        'Series': 'Series'
    }
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Homes',
    legend_title='Series',
    template='plotly_white'
)

fig.show()


**Seattle Listings and Sales vs National Average (supports Question 1):**  
This visualization answers **Question 1** — *Are there seasonal patterns in listings counts, sales counts, and price for houses in the Seattle metro area that could inform my purchase timing?* (This chart covers **listings and sales**; median price is shown in the chart below.)

**Why this chart type:** I used a **multi-line time series** with four series on one chart so the reader can compare **Seattle listings and sales** directly against **national averages** on the same monthly timeline. A line chart is appropriate here because the data are **counts over time**. It makes repeating **seasonal ups and downs** visible (winter dips, summer peaks) as well as longer-run changes (for example, the 2021–2022 market shift). Plotting all four lines together makes it easy to see whether Seattle follows the national rhythm and whether its swings are **larger or smaller**.

**What I want the reader to take away:** Seattle shows a **strong seasonal cycle**: listings tend to peak in **late summer** and bottom in **winter**, while sales peak slightly earlier in **late spring/early summer** and are lowest in **winter** — matching the Section 3 pandas analysis (~+25% listings in August vs ~−33% in February; sales peak ~June). Seattle’s seasonality is **stronger than the national average** on this chart. For purchase timing, that suggests **more inventory but more competition in summer**, and **fewer homes/transactions but potentially less competition in winter**.

In [26]:
# Median sale price — Seattle, national average, and West Coast metros
import pandas as pd
import plotly.express as px

metro_price = pd.read_csv('Metro_median_sale_price_now_uc_sfrcondo_month.csv')

date_cols = [c for c in metro_price.columns if c.startswith('20') and '-' in c]

metro_long = metro_price.melt(
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'],
    value_vars=date_cols,
    var_name='Date',
    value_name='MedianPrice',
)

metro_long['Date'] = pd.to_datetime(metro_long['Date'])
metro_long['MedianPrice'] = pd.to_numeric(metro_long['MedianPrice'], errors='coerce')

start_date = pd.Timestamp('2016-01-01')
metro_long = metro_long[metro_long['Date'] >= start_date]

COMPARE_METROS = {
    'Seattle, WA': 'Seattle Median',
    'San Francisco, CA': 'San Francisco Median',
    'Portland, OR': 'Portland Median',
    'San Diego, CA': 'San Diego Median',
}

metro_lines = metro_long[
    (metro_long['RegionName'].isin(COMPARE_METROS.keys()))
    & (metro_long['RegionType'] == 'msa')
].copy()
metro_lines['Series'] = metro_lines['RegionName'].map(COMPARE_METROS)

national_median = (
    metro_long[metro_long['RegionType'] != 'country']
    .groupby('Date', as_index=False)['MedianPrice']
    .mean()
)
national_median['Series'] = 'National Median'

plot_df = pd.concat(
    [
        metro_lines[['Date', 'MedianPrice', 'Series']],
        national_median[['Date', 'MedianPrice', 'Series']],
    ],
    ignore_index=True,
)

series_order = [
    'Seattle Median',
    'National Median',
    'San Francisco Median',
    'Portland Median',
    'San Diego Median',
]

fig = px.line(
    plot_df.sort_values(['Series', 'Date']),
    x='Date',
    y='MedianPrice',
    color='Series',
    category_orders={'Series': series_order},
    title='Median Sale Price: Seattle and West Coast Metros vs National Average (Last 10 Years)',
    labels={
        'Date': 'Date',
        'MedianPrice': 'Median Sale Price ($)',
        'Series': 'Series',
    },
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Median Sale Price ($)',
    legend_title='Series',
    template='plotly_white',
)

fig.show()

**Median Sale Price: Seattle and West Coast Metros (supports Question 1):**  
This visualization also supports **Question 1** — *Are there seasonal patterns in listings counts, sales counts, and price for houses in the Seattle metro area that could inform my purchase timing?* (This chart focuses on **median sale price**; listings and sales are in the chart above.)

**Why this chart type:** I used a **multi-line time series** to compare **Seattle** with **San Francisco, Portland, San Diego**, and a **national metro average** on the same monthly scale. That lets the reader see both **how expensive Seattle is relative to peers** and **how prices move over time**, including repeating seasonal bumps and longer market shifts (such as the rapid rise around 2021–2022 and the cooldown afterward). A line chart works here because price is a continuous monthly series so is better for spotting gradual trends and mild seasonality than a bar chart.

**What I want the reader to take away:** **Price seasonality is much weaker than listings/sales seasonality.** Seattle’s median sale price is somewhat **higher in late spring/early summer** and **lower in winter** (~9% below peak in the Section 3 analysis), but the line is **flatter** than the listings/sales chart. Seattle generally sits **below San Francisco** and **above Portland** on this chart, closer to San Diego depending on the year. For purchase timing, this suggests **winter may offer slightly lower typical prices**, but the effect is **modest compared to how much activity swings** so timing on price alone matters less than the inventory/competition tradeoffs shown in the listings and sales chart.

In [39]:
# Your visualization
import pandas as pd
import plotly.express as px

# Step 1: Load the 2-bedroom zip-level ZHVI data
df_2bed = pd.read_csv('Zip_zhvi_bdrmcnt_2_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (1).csv')

# Identify the date columns (from 2000-01-31 to 2026-03-31)
date_cols = [col for col in df_2bed.columns if col.startswith('20') and '-' in col]

# Melt the dataframe to long format
df_long = df_2bed.melt(
    id_vars=['RegionID', 'RegionName', 'RegionType', 'City', 'State'],
    value_vars=date_cols,
    var_name='Date',
    value_name='ZHVI'
)

# Convert Date to datetime
df_long['Date'] = pd.to_datetime(df_long['Date'])

# Filter for the last 10 years (2016 onwards)
df_filtered = df_long[df_long['Date'] >= '2016-01-01']

# Step 2: Filter for comparison regions
bellevue_overall = df_filtered[df_filtered['City'] == 'Bellevue'].copy()
bellevue_overall['Region_Group'] = 'Bellevue (Overall)'

bellevue_98004 = df_filtered[(df_filtered['RegionName'] == '98004') & (df_filtered['RegionType'] == 'zip')].copy()
bellevue_98004['Region_Group'] = 'Bellevue (98004)'

seattle_city = df_filtered[(df_filtered['City'] == 'Seattle') & (df_filtered['RegionType'] == 'city')].copy()
seattle_city['Region_Group'] = 'Seattle (City)'

portland_city = df_filtered[(df_filtered['City'] == 'Portland') & (df_filtered['RegionType'] == 'city')].copy()
portland_city['Region_Group'] = 'Portland (City)'

washington_state = df_filtered[(df_filtered['State'] == 'WA') & (df_filtered['RegionType'] == 'zip')].copy()
washington_state['Region_Group'] = 'Washington (Zip Average)'

oregon_state = df_filtered[(df_filtered['State'] == 'OR') & (df_filtered['RegionType'] == 'zip')].copy()
oregon_state['Region_Group'] = 'Oregon (Zip Average)'

# Combine the filtered data
df_plot = pd.concat([
    bellevue_overall,
    bellevue_98004,
    seattle_city,
    portland_city,
    washington_state,
    oregon_state
], ignore_index=True)

# Step 3: Aggregate by date and region group (average ZHVI per month)
df_agg = df_plot.groupby(['Date', 'Region_Group'])['ZHVI'].mean().reset_index()

# Step 4: Create the line chart
fig = px.line(
    df_agg,
    x='Date',
    y='ZHVI',
    color='Region_Group',
    title='2-Bedroom Home Value Trends Over the Last 10 Years: Bellevue, Seattle, Portland, and WA/OR Averages',
    labels={
        'Date': 'Date',
        'ZHVI': 'ZHVI (2-Bedroom Median Home Value Estimate)',
        'Region_Group': 'Region'
    }
)

# Customize the layout
fig.update_layout(
    xaxis_title='Year',
    yaxis_title='ZHVI (2-Bedroom Median Home Value Estimate)',
    legend_title='Region',
    template='plotly_white'
)

# Show the chart
fig.show()


This visualization supports Question 1 by showing how 2-bedroom home values (Zillow Home Value Index) changed over the last decade — important price context for purchase timing — and adds neighborhood context for Question 2 by comparing Bellevue (98004) to nearby benchmarks.

Why this chart type: I used a multi-line time series at monthly intervals to compare Bellevue (98004), Bellevue overall, Seattle, Portland, and state zip averages (WA/OR) on one timeline. A line chart fits because ZHVI is a continuous estimate over time; it shows both the long-run trend (especially the sharp rise around 2021–2022) and how local values track or diverge from regional benchmarks. Using zip/city-level ZHVI also lets me keep a 2-bedroom, middle-tier focus closer to my neighborhood than metro-wide median sale price alone.

What I want the reader to take away: Home values rose substantially across all regions on this chart from 2016–2026, with a steep climb in the early 2020s and some leveling afterward. Bellevue (98004) ends the period well above Portland and the Oregon zip average, and generally above or comparable to Seattle depending on the year — suggesting a high-value local market relative to peers. For purchase timing (Question 1), this reinforces that price level and trend matter alongside seasonality: even when listings/sales are seasonal, underlying home values shifted dramatically over the decade, so timing is not only about when in the year, but also where the market was in its broader cycle.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  

The most important takeaway from this analysis is that **purchase timing in Seattle is driven more by market activity than by price swings**: listings and sales show **strong seasonality**, but **median sale prices are much flatter** — so winter may mean less competition, not dramatically cheaper homes. What surprised me most was the **ZORI vs ZORDI split**: **rent levels rose** in Seattle metro while **rental listing engagement (ZORDI) fell**, suggesting rising costs without matching renter demand — and that **Bellevue (98004) ZORI growth lagged** metro and national benchmarks even as home values climbed sharply. Taken together, this points to a **challenging market to enter** right now for both buyers and renters. If I had more time and data, I would **track whether more homes come onto the market** (especially as inventory shifts) and **continue monitoring policy changes**, including **Washington state income tax proposals** that could potentially expand to lower brackets. Either of these could change affordability and demand. A key limitation is that these datasets **do not cover the same geography or housing type consistently** (for example, ZORDI is metro-level and condo-focused, listings/sales are metro-only, and ZHVI is zip-level 2-bedroom estimates), so I **cannot conclude causation, individual affordability, or the best month to buy** and only describe patterns in Zillow’s published indexes over time.

---

Competency Claims
C5 — Data Analysis with Pandas
I used pandas to answer three specific research questions about Zillow housing data in Section 3. For Question 1 (seasonality), I filtered Seattle metro rows, melted wide monthly CSVs into long format, used groupby on calendar month to compute average listings/sales/median price, and compared Seattle to a national groupby('Date').mean() benchmark. For Question 2, I calculated total ZORI growth (%) for Bellevue (98004), Seattle metro, and the U.S., then pivoted annual year-over-year changes to match my bar chart. For Question 3, I merged aligned ZORI and ZORDI monthly series and computed correlation after restricting to June 2020 onward (when both series exist). In each case I wrote an interpretation of what the numbers mean for the market not just the printed tables.

C6 — Data Visualization
I built multiple Plotly charts that each support a specific argument: a four-line time series for Seattle listings/sales vs national averages (Question 1 seasonality), a dual-axis line chart for ZORI vs ZORDI divergence (Question 3), a line + grouped bar panel for 10-year ZORI growth (Question 2), and additional metro comparison charts for median sale price and 2-bedroom ZHVI trends. Below each chart I added a markdown cell explaining why that chart type fit the data structure and question and what I want the reader to take away — for example, that price seasonality is much weaker than activity seasonality, so purchase timing is a tradeoff between inventory and competition, not a clear “cheapest month.” 

C7 — Critical Evaluation and Professional Judgment
I did not treat AI-generated code or summaries as final without checking them against the data. When Cursor produced analysis cells that did not actually appear in my notebook (the placeholder # Your analysis for Question 2 remained), I caught the gap, re-ran the workflow, and confirmed the cells executed and matched my charts before submitting. I also corrected the analysis scope when the data did not match my original questions. For example, I adjusted Question 3 to Seattle metro and a June 2020 start date because ZORDI is not available at zip level, and revising my overview when I learned listings/sales are metro-only while ZHVI is zip-level 2-bedroom. 